# The Perceptron
The simplest perceptron is just a linear threshold function. Which means:\
$x \in \mathbb{R}^{d}$, choose wieghts $w \in \mathbb{R^d}$ and a bias $b \in \mathbb{R}$ and we define:\
$$f(x) = 
     \begin{cases}
       \text{1,} &\quad\text{if $w^Tx + b >0$ }\\
       \text{0,} &\quad\text{otherwise} \\
     \end{cases}
$$
My previous implementation of a simple perceptron for CIUFLA 2025 was the code present in the second code cell bellow.

In [1]:
import pandas as pd
import numpy as np
import math
from typing import Iterable, List, Sequence
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

In [2]:
class Perceptron:
    def __init__(self, n_features):
        self.w = [0.0] * n_features
        self.b = 0.0

    def score(self, x):
        return sum(wi * xi for wi, xi in zip(self.w, x)) + self.b

    def predict(self, x):
        """
        Idea
            f(x) = 1 if w^tx + b > 0
                   0 otherwise
        """
        return 1 if self.score(x) > 0 else -1

    def fit(self, X, y, epochs=10):
        for _ in range(epochs):
            for xi, yi in zip(X, y):
                if yi * self.score(xi) <= 0:
                    self.w = [wi + yi * xij for wi, xij in zip(self.w, xi)]
                    self.b += yi

# A Quantum Perceptron
So now, given the motivation arised by reading the Turing paper, what follows is my implementation of a quantum perpectron.

In [3]:
class QPerceptron:
    """
    Idea:
        score(x) = w·x + b
        theta = clip(score(x), -pi/2, pi/2)
        prepare |0>, apply Ry(theta), then measure <X>.
        Since <X> = sin(theta), prediction is sign(<X>).

    Labels are assumed to be in {-1, +1}.
    Training uses the same mistake-driven update as the classical perceptron.
    """

    def __init__(self, n_features: int):
        self.w = [0.0] * n_features
        self.b = 0.0
        self.observable = SparsePauliOp.from_list([("X", 1.0)])

    def linear_score(self, x: Sequence[float]) -> float:
        return sum(wi * xi for wi, xi in zip(self.w, x)) + self.b

    def _angle(self, x: Sequence[float]) -> float:
        s = self.linear_score(x)
        return max(-math.pi / 2, min(math.pi / 2, s))

    def _circuit(self, x: Sequence[float]) -> QuantumCircuit:
        theta = self._angle(x)
        qc = QuantumCircuit(1)
        qc.ry(theta, 0)
        return qc

    def quantum_score(self, x: Sequence[float]) -> float:
        """
        Returns <X> for the 1-qubit state after Ry(theta(x)).
        """
        qc = self._circuit(x)
        psi = Statevector.from_instruction(qc)
        value = psi.expectation_value(self.observable)
        return float(value.real)

    def predict(self, x: Sequence[float]) -> int:
        return 1 if self.quantum_score(x) > 0 else -1

    def fit(
        self,
        X: Iterable[Sequence[float]],
        y: Iterable[int],
        epochs: int = 10,
    ) -> None:
        """
        Classical perceptron-style training on the underlying affine score.
        """
        X = list(X)
        y = list(y)

        for _ in range(epochs):
            for xi, yi in zip(X, y):
                if yi * self.linear_score(xi) <= 0:
                    self.w = [wi + yi * xij for wi, xij in zip(self.w, xi)]
                    self.b += yi

In [4]:
def test_perceptron():
    X = [
        [2.0, 1.0],
        [1.0, 2.0],
        [-1.0, -1.0],
        [-2.0, -1.0],
    ]
    y = [1, 1, -1, -1]

    p = Perceptron(n_features=2)
    p.fit(X, y, epochs=10)

    print("weights:", p.w)
    print("bias:", p.b)

    x1 = [3.0, 1.0]
    x2 = [-1.0, -2.0]
    
    print("prediction for", x1, "=", p.predict(x1))
    print("prediction for", x2, "=", p.predict(x2))

def test_qperceptron():
    X = [
        [2.0, 1.0],
        [1.0, 2.0],
        [-1.0, -1.0],
        [-2.0, -1.0],
    ]
    y = [1, 1, -1, -1]

    qp = QPerceptron(n_features=2)
    qp.fit(X, y, epochs=10)

    print("weights:", qp.w)
    print("bias:", qp.b)

    x1 = [3.0, 1.0]
    x2 = [-1.0, -2.0]

    print("linear score", x1, "=", qp.linear_score(x1))
    print("quantum score", x1, "=", qp.quantum_score(x1))
    print("prediction", x1, "=", qp.predict(x1))

    print("linear score", x2, "=", qp.linear_score(x2))
    print("quantum score", x2, "=", qp.quantum_score(x2))
    print("prediction", x2, "=", qp.predict(x2))

In [5]:
test_perceptron()

weights: [2.0, 1.0]
bias: 1.0
prediction for [3.0, 1.0] = 1
prediction for [-1.0, -2.0] = -1


In [6]:
test_qperceptron()

weights: [2.0, 1.0]
bias: 1.0
linear score [3.0, 1.0] = 8.0
quantum score [3.0, 1.0] = 1.0
prediction [3.0, 1.0] = 1
linear score [-1.0, -2.0] = -3.0
quantum score [-1.0, -2.0] = -1.0
prediction [-1.0, -2.0] = -1


# Notes
A Perceptron, on the usual sense, appers to be not equivalent to a quantum machine, the results are the same. The question that I would like to address is why? It appers to be a linear algebra problem (in quantum mechanics/information theory).\
This paper Wiebe, N., Kapoor, A., & Svore, K. M. (2016). Quantum perceptron models. arXiv preprint arXiv:1602.04799.\
Appers to be a good starting point to dive on. What follows next is trying to understand and explain this paper (implementing it and doing the math/physics).